# Billing cost model

Shows the work behind [`docs/BILLING_COST_MODEL.md`](../docs/BILLING_COST_MODEL.md):
every per-tier cap and price in the product traced back to a vendor rate times a
usage assumption.

**Three inputs, all editable:**

| File | What it holds |
| --- | --- |
| [`pricing.json`](pricing.json) | vendor unit prices, keyed by model and voice family |
| [`assumptions.json`](assumptions.json) | how much of each service a lecture and its audience consume |
| [`../config/plans.json`](../config/plans.json) | the caps actually shipped — what we are evaluating |

Change a number, `Run All`, read the new answer. Nothing is hard-coded in the
notebook; stdlib only, so it runs in any kernel.

> Prices were verified 2026-07-31 against the vendors' own pricing pages —
> §1 lists every source URL. The usage assumptions are educated guesses; the
> ones marked `weak` in `assumptions.json` are what a semester of pilot data
> should replace.


## 1. Load the inputs


In [ ]:
# Read the three data files into memory. Nothing here computes anything yet —
# it just loads the numbers and defines two small helpers used throughout.
import json, os
from IPython.display import Markdown, display

ROOT = os.path.abspath('..')          # the repo root, one level up from cost-model/
P = json.load(open('pricing.json'))   # what each service charges
A = json.load(open('assumptions.json'))  # how much of it we expect to use

# Caps are read from the SHIPPED config rather than copied into this folder, so
# the notebook always evaluates the plans customers actually get. Edit that file
# and re-run to see a cap change ripple through every number below.
PLANS = json.load(open(os.path.join(ROOT, 'config', 'plans.json')))


def table(headers, rows, title=None):
    """Prints a list of rows as a markdown table.

    Jupyter renders markdown output, so building a table as text and handing it
    to display() gives readable output without pulling in pandas — which keeps
    this notebook runnable in any Python kernel.
    """
    out = (f'**{title}**\n\n' if title else '')
    out += '| ' + ' | '.join(headers) + ' |\n'      # header row
    out += '|' + '|'.join(['---'] * len(headers)) + '|\n'  # the ---- separator
    for r in rows:
        out += '| ' + ' | '.join(str(c) for c in r) + ' |\n'
    display(Markdown(out))


def usd(x, dp=2):
    """Formats a number as dollars: 1234.5 -> '$1,234.50'.

    dp raises the decimal places for rates so small that 2dp would round them to
    zero — $0.016 per STT minute, for instance.
    """
    return f'${x:,.{dp}f}'


print(f"pricing asOf {P['asOf']} | tiers {list(PLANS)} | default model {P['ai']['defaultModel']}")


### Where these prices come from

Every rate in `pricing.json` traces to a vendor pricing page. Rendered from the
data rather than typed here, so adding a price and its source keeps them
together — and `verified` says when a human last read the page, which is the
number that goes stale.


In [ ]:
# pricing.json carries a _sources block: one entry per vendor with a URL, the
# date someone last checked it, and any billing quirk worth remembering. Turn
# each entry into a table row, with the service name as a clickable link.
rows = [[f"[{v['service']}]({v['url']})", v['verified'], v.get('note', '')]
        for v in P['_sources'].values()]
table(['Service', 'Verified', 'Gotchas worth remembering'], rows,
      'Pricing sources')

# Some viewers (plain nbconvert output, a diff, a terminal) swallow markdown
# links, so print the bare URLs as well.
print('Raw URLs, for when the markdown links are not clickable:')
for k, v in P['_sources'].items():
    print(f"  {k:12s} {v['url']}")


### Drift check

`pricing.json` is the modelling copy so you can experiment freely. The server
reads [`../config/service-prices.json`](../config/service-prices.json). If the two
diverge, the model is describing a product we do not ship — so check.


In [ ]:
# The server bills against its own copy of these prices. If someone edits one
# file and forgets the other, this model quietly starts describing a product we
# do not sell — so compare them every run and say so loudly.
live = json.load(open(os.path.join(ROOT, 'config', 'service-prices.json')))


def flat(d, prefix=''):
    """Flattens nested dicts into one level: {'stt': {'perMin': 1}} -> {'stt.perMin': 1}.

    Comparing two nested structures key-by-key is fiddly; comparing two flat
    dictionaries is trivial. Keys starting with '_' are documentation (notes,
    sources) rather than prices, so they are skipped.
    """
    out = {}
    for k, v in d.items():
        if k.startswith('_'):
            continue
        key = f'{prefix}{k}'
        # A nested dict recurses; anything else is a leaf value we can compare.
        out.update(flat(v, key + '.') if isinstance(v, dict) else {key: v})
    return out


a, b = flat(P), flat(live)
# Union of both key sets, so a price present in only one file also shows up.
drift = [(k, a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)]
if drift:
    table(['key', 'pricing.json', 'config/service-prices.json'], drift,
          'DRIFT — the model and the server disagree')
else:
    print('No drift: modelling prices match the shipped config.')


## 2. Unit prices

Pulled by name, so switching `GEMINI_MODEL` or a narration voice family in the
config changes the model too.


In [ ]:
# Pick the prices for whichever model and voice families are configured, rather
# than hard-coding today's choices. Swap 'defaultModel' in pricing.json and every
# figure below re-prices itself.
model = P['ai']['models'][P['ai']['defaultModel']]

# Vendors quote per MILLION tokens or characters. We count single tokens and
# characters, so divide by 1e6 once here — then every later line is just
# 'how many' x 'price each', with no stray millions to keep track of.
TOK_IN   = model['inputPerMillionTokens'] / 1e6   # prompt tokens we send
TOK_OUT  = model['outputPerMillionTokens'] / 1e6  # tokens the model writes back
EMBED    = list(P['ai']['embeddingModels'].values())[0]['inputPerMillionTokens'] / 1e6
IMAGE    = list(P['ai']['imageModels'].values())[0]['perImage']

# Speech-to-Text V2 has ONE rate covering live streaming and standard batch, so
# diarization (which runs as a batch job) costs the same per minute as capturing
# the lecture live. The cheaper 'dynamic batch' rate needs 24h turnaround, which
# the per-slide 'identify speakers' button cannot wait for.
STT      = P['stt']['recognitionPerMinute']

# Two TTS voice families: the everyday one and the premium one, ~2x the price.
TTS_STD  = P['tts']['voiceFamilies'][P['tts']['defaultStandardFamily']]['perMillionChars'] / 1e6
TTS_PREM = P['tts']['voiceFamilies'][P['tts']['defaultPremiumFamily']]['perMillionChars'] / 1e6
TRANSLATE = P['translation']['perMillionChars'] / 1e6
GIB_MONTH, EGRESS = P['storage']['perGibMonth'], P['storage']['egressPerGib']

# Stripe takes a percentage AND a flat fee per charge. The flat fee is why a $19
# plan keeps proportionally less of its price than a $299 one.
PAY_RATE  = P['payments']['rate'] + P['payments']['billingRate']
PAY_FIXED = P['payments']['perTransaction']

# Multiply back up by 1e6 purely for display, so the table reads in the same
# units the vendor quotes.
src = lambda key: f"[source]({P['_sources'][key]['url']})"
table(['Unit', 'Price', 'Source'], [
    [f"{P['ai']['defaultModel']} input",  f'{usd(TOK_IN*1e6)} / 1M tokens', src('gemini')],
    [f"{P['ai']['defaultModel']} output", f'{usd(TOK_OUT*1e6)} / 1M tokens', src('gemini')],
    ['STT recognition (streaming + batch)', f'{usd(STT,3)} / min', src('stt')],
    ['STT dynamic batch (24h turnaround)', f"{usd(P['stt']['dynamicBatchPerMinute'],3)} / min", src('stt')],
    [f"TTS {P['tts']['defaultStandardFamily']}",  f'{usd(TTS_STD*1e6)} / 1M chars', src('tts')],
    [f"TTS {P['tts']['defaultPremiumFamily']}", f'{usd(TTS_PREM*1e6)} / 1M chars', src('tts')],
    ['Translation', f'{usd(TRANSLATE*1e6)} / 1M chars', src('translation')],
    ['Storage / egress', f'{usd(GIB_MONTH,3)} per GiB-month / {usd(EGRESS,3)} per GiB', src('spaces')],
    ['Payments', f'{PAY_RATE*100:.1f}% + {usd(PAY_FIXED)}', src('stripe')],
], 'Unit prices in play')


## 3. One lecture

Everything scales from **lecture duration**. Change
`assumptions.lecture.durationMinutes` and every figure below moves with it.


In [ ]:
def per_lecture(duration=None):
    """Works out what a single lecture costs us, line by line.

    Three phases, which cost money in quite different ways:

      1. LIVE     — the instructor is speaking. Speech is transcribed and every
                    finalised phrase triggers one AI call that writes a slide.
      2. POST     — afterwards: tidying slides, generating narration audio,
                    optionally working out who spoke (diarization).
      3. AUDIENCE — students watching later, and any translations they ask for.

    Pass a duration to model a longer or shorter lecture; everything scales with
    it, because a 2x longer lecture means ~2x the phrases, slides and audio.
    """
    L, X, R, AU = A['lecture'], A['perLectureExtras'], A['revision'], A['audience']
    d = duration or L['durationMinutes']
    scale = d / L['durationMinutes']   # 1.0 for a standard lecture, 1.47 for 110 min
    slides = L['slidesPerLecture'] * scale
    calls  = L['phrasesPerMinute'] * d  # one AI call per finalised phrase of speech

    # --- 1. LIVE -----------------------------------------------------------
    # Each call sends the prompt plus context (input tokens) and gets a slide
    # back (output tokens). Output is 6x dearer per token, but there is far less
    # of it, so input still dominates.
    generation = calls * (L['inputTokensPerCall'] * TOK_IN + L['outputTokensPerCall'] * TOK_OUT)

    # Smaller AI jobs: picking the best stock image per slide, writing the quiz
    # once per lecture, and embeddings that keep whiteboard marks anchored.
    rerank = slides * (X['rerankInputTokensPerSlide'] * TOK_IN + X['rerankOutputTokensPerSlide'] * TOK_OUT)
    quiz = X['quizInputTokens'] * TOK_IN + X['quizOutputTokens'] * TOK_OUT
    embeddings = X['embeddingTokens'] * scale * EMBED

    # Cloud transcription, charged per minute of audio. Zero on the browser
    # engine, which is free — the single biggest lever in this whole model.
    stt = d * L['sttMinutesPerLectureMinute'] * STT

    # --- 2. POST-LECTURE ---------------------------------------------------
    # The 'light' profile: an average teacher tidies a quarter of the slides,
    # not all of them.
    refined = slides * R['refinedSlideShare']
    refine = refined * (R['refineInputTokensPerSlide'] * TOK_IN + R['refineOutputTokensPerSlide'] * TOK_OUT)
    narrate = refined * (R['narrateInputTokensPerSlide'] * TOK_IN + R['narrateOutputTokensPerSlide'] * TOK_OUT)

    # Turning slide narration into speech, charged per character of text sent.
    narration_chars = slides * L['narrationCharsPerSlide']
    tts = narration_chars * TTS_STD
    # Editing a slide's words invalidates its cached audio, so it is synthesised
    # again — which is why revision volume matters more than it looks.
    resynth = R['resynthesizedSlides'] * L['narrationCharsPerSlide'] * TTS_STD

    # Re-transcribing an edited clip goes through the LIVE recogniser, so it is
    # billed at the streaming rate rather than a cheaper batch one.
    retranscribe = R['retranscribeMinutes'] * STT
    # Diarization = working out who spoke when. Only some lectures get it, but
    # it re-processes the whole recording at the full per-minute rate.
    diarize = R['diarizeShareOfLectures'] * d * STT

    # --- 3. AUDIENCE -------------------------------------------------------
    # Cost here scales with LANGUAGES, not viewers: the first student to ask for
    # a translation pays for it, everyone after that reads the cached copy.
    locales = AU['localesPerDeck']
    translate = locales * slides * L['slideTextChars'] * TRANSLATE
    translated_tts = locales * narration_chars * TTS_STD  # the translation needs narrating too

    # Playing an existing deck only moves bytes out of storage — fractions of a
    # cent. /1024 converts megabytes to gigabytes, which is how egress is priced.
    playbacks = AU['studentsPerSection'] * AU['deckOpenRate'] * AU['playbacksPerViewingStudent']
    egress = playbacks * (AU['imageMbPerPlayback'] + AU['narrationAudioMbPerPlayback']) / 1024 * EGRESS

    # Retained audio: sample rate x bytes per sample x 60 seconds = bytes per
    # minute; /1e6 makes megabytes. Storage is then billed per GiB per month.
    audio_mb = d * L['captureSampleRateHz'] * L['captureBytesPerSample'] * 60 / 1e6
    storage = audio_mb / 1024 * GIB_MONTH

    lines = {
        'Slide generation': generation, 'Image re-rank': rerank,
        'Quiz generation': quiz, 'Embeddings': embeddings,
        'Cloud STT (live)': stt, 'Refine + narrate passes': refine + narrate,
        'TTS narration': tts, 'TTS re-synthesis': resynth,
        'Cloud STT (re-transcribe)': retranscribe, 'Diarization': diarize,
        'Translation (audience)': translate,
        'Translated narration (audience)': translated_tts,
        'Playback egress (audience)': egress, 'Audio storage': storage,
    }
    return {'lines': lines, 'slides': slides, 'audioMb': audio_mb,
            'aiTokens': calls * (L['inputTokensPerCall'] + L['outputTokensPerCall']),
            'narrationChars': narration_chars, 'playbacks': playbacks,
            # Browser capture has no cloud STT — and keeps no audio, so there is
            # nothing to diarize or re-transcribe either. Subtracting this one
            # figure converts any total to its browser-tier equivalent.
            'cloudOnly': stt + retranscribe + diarize}


r = per_lecture()
total = sum(r['lines'].values())
table(['Line', 'Cost'], [[k, usd(v, 3)] for k, v in r['lines'].items()],
      f"One {A['lecture']['durationMinutes']}-minute lecture")
print(f"{r['slides']:.0f} slides | {r['aiTokens']/1e6:.2f}M tokens | "
      f"{r['narrationChars']:,.0f} narration chars | {r['audioMb']:.0f} MB audio | "
      f"{r['playbacks']:.0f} playbacks")
print(f"TOTAL {usd(total)} with cloud capture | {usd(total - r['cloudOnly'])} on browser capture")


## 4. Per 100 lectures, by service

The fastest read: which services actually cost money, and how much of it the
audience drives.


In [ ]:
# Per-lecture figures are fractions of a cent, which are hard to compare at a
# glance. Regroup the same numbers by SERVICE and scale to 100 lectures, so the
# ranking is obvious and the units are dollars people can reason about.
L = r['lines']
svc = {
    'Cloud STT': L['Cloud STT (live)'] + L['Cloud STT (re-transcribe)'],
    'Gemini (all LLM)': L['Slide generation'] + L['Image re-rank'] + L['Quiz generation']
                        + L['Embeddings'] + L['Refine + narrate passes'],
    'TTS': L['TTS narration'] + L['TTS re-synthesis'] + L['Translated narration (audience)'],
    'Diarization': L['Diarization'],
    'Translation': L['Translation (audience)'],
    'Storage + egress': L['Audio storage'] + L['Playback egress (audience)'],
}
svc = {k: v * 100 for k, v in svc.items()}   # one lecture -> one hundred
tot = sum(svc.values())

# Which share of the bill is caused by students rather than the instructor?
# Only work a VIEWER triggers counts: a new translation, its narration, and the
# bytes shipped during playback.
student = (L['Translation (audience)'] + L['Translated narration (audience)']
           + L['Playback egress (audience)']) * 100

table(['Service', 'Cost / 100 lectures', 'Share'],
      [[k, usd(v), f'{v/tot*100:.1f}%'] for k, v in sorted(svc.items(), key=lambda x: -x[1])]
      + [['**Total**', f'**{usd(tot)}**', '']],
      'Per 100 lectures')
print(f"Browser-capture tiers: {usd(tot - r['cloudOnly']*100)} "
      f"({(r['cloudOnly']*100)/tot*100:.0f}% of cost removed with one config value)")
print(f"Instructor-driven {usd(tot-student)} ({(tot-student)/tot*100:.0f}%) | "
      f"student-driven {usd(student)} ({student/tot*100:.0f}%)")


## 5. What each tier costs

Reads the **shipped** caps from `config/plans.json`, so editing a cap there and
rerunning shows its price impact immediately.

An unlimited cap (`null`) contributes nothing — an unbounded cap has no worst
case, which is exactly why no tier has one.

Four utilisation levels, since subscribers are not all the same shape:

- **Light** — subscribed for the capability rather than the volume: a few
  lectures a month, little revision. Common, and the most profitable.
- **Expected** — the planning case, half the caps.
- **Heavy** — a power user running close to the limits.
- **At caps** — everything consumed. Rare, but it is the number the price has
  to survive.


In [ ]:
# Roughly 93% of the tokens we spend are input (prompts and context) and only
# 7% are output. Blending the two rates in that proportion gives one number we
# can multiply a raw token cap by.
BLEND_INPUT = 0.93
AI_BLENDED = (BLEND_INPUT * model['inputPerMillionTokens']
              + (1 - BLEND_INPUT) * model['outputPerMillionTokens']) / 1e6

# One translated deck costs this many characters of translation.
SLIDE_TEXT_PER_DECK = A['lecture']['slidesPerLecture'] * A['lecture']['slideTextChars']


def worst_case(caps):
    """What a subscriber would cost us if they used up every one of their caps.

    This is the number a price has to survive. Each cap is a quantity (tokens,
    minutes, characters); multiplying by that service's unit price turns it into
    dollars. `caps.get(k) or 0` treats both a missing cap and an unlimited one
    (null) as zero — an unlimited cap has no worst case, which is precisely why
    we stopped shipping them.
    """
    v = lambda k: caps.get(k) or 0
    return {
        'aiTokens': v('aiTokens') * AI_BLENDED,
        'sttMinutes': v('sttMinutes') * STT,
        'diarizationMinutes': v('diarizationMinutes') * STT,
        'ttsCharacters': v('ttsCharacters') * TTS_STD,
        'ttsPremiumCharacters': v('ttsPremiumCharacters') * TTS_PREM,
        'aiImages': v('aiImages') * IMAGE,
        'translationCharacters': v('translationCharacters') * TRANSLATE,
        'audienceTtsCharacters': v('audienceTtsCharacters') * TTS_STD,
        # This cap counts whole decks translated, so convert to characters first.
        'audienceLocales': v('audienceLocales') * SLIDE_TEXT_PER_DECK * TRANSLATE,
        'audioStorageMb': v('audioStorageMb') / 1024 * GIB_MONTH,
    }


# The lowest price that still covers the service cost: add Stripe's flat fee,
# then divide by what is left after their percentage. Solves price - fees = cost.
floor_price = lambda cost: (cost + PAY_FIXED) / (1 - PAY_RATE)

TG = A['targets']
rows, breakdown = [], {}
for tier, plan in PLANS.items():
    parts = worst_case(plan['caps'])
    breakdown[tier] = parts        # keep the per-metric split for the next table
    wc = sum(parts.values())
    price = A['tiers'][tier]['priceUsd']
    # Lighter users cost a fraction of the worst case — same caps, less used.
    rows.append([tier, A['tiers'][tier]['lecturesPerMonth'],
                 usd(wc * TG['lightUtilisation']), usd(wc * TG['expectedUtilisation']),
                 usd(wc * TG['heavyUtilisation']), usd(wc), usd(floor_price(wc)),
                 usd(price), f'{wc/price*100:.0f}%' if price else '—',
                 '✅' if not price or wc/price <= TG['worstCaseShareOfPrice'] + 0.15 else '⚠️'])
table(['Tier', 'Lectures/mo', f"Light ({TG['lightUtilisation']:.0%})",
       f"Expected ({TG['expectedUtilisation']:.0%})", f"Heavy ({TG['heavyUtilisation']:.0%})",
       'At caps', 'Price floor', 'Price', 'Maxed as % of price',
       f"vs {TG['worstCaseShareOfPrice']:.0%} target"], rows,
      'Per-tier cost at four utilisation levels, from the shipped caps')

# Same worst-case numbers, split by metric, so it is clear WHICH cap is
# expensive rather than just that the tier is.
table(['Metric'] + list(PLANS),
      [[m] + [usd(breakdown[t][m]) for t in PLANS] for m in breakdown['pro']],
      'Where each tier\'s worst case comes from')


## 6. Break-even

Fixed costs are the same whatever we charge, which is why a cheap tier needs so
many more subscribers than an expensive one.


In [ ]:
# Fixed costs are what we pay monthly regardless of how many customers exist:
# servers, database, storage, developer tooling. Subscriptions have to cover
# these on top of their own service usage.
for scenario, costs in A['fixedCostsUsdPerMonth'].items():
    fixed = sum(costs.values())
    rows = [[k, usd(v)] for k, v in costs.items()] + [['**Total**', f'**{usd(fixed)}**']]
    table(['Line', 'Per month'], rows, f'Fixed costs — {scenario}')

    out = []
    for tier, plan in PLANS.items():
        price = A['tiers'][tier]['priceUsd']
        if not price:
            continue          # the free tier contributes nothing by definition
        wc = sum(worst_case(plan['caps']).values())
        fee = PAY_RATE * price + PAY_FIXED

        def subs(c):
            """How many subscribers at this usage level to cover the fixed costs.

            Each one contributes price - their service cost - payment fees. If
            that is zero or negative the tier never pays for itself at that usage
            level, hence the infinity. The -(-x // y) idiom rounds UP, because
            two-thirds of a subscriber does not pay a bill.
            """
            contribution = price - c - fee
            return '∞' if contribution <= 0 else f'{-(-fixed // contribution):.0f}'

        out.append([tier, usd(price), subs(wc * TG['lightUtilisation']),
                    subs(wc * TG['expectedUtilisation']),
                    subs(wc * TG['heavyUtilisation']), subs(wc)])
    table(['Tier', 'Price', 'Light use', 'Expected use', 'Heavy use', 'At caps'], out,
          f'Subscribers needed to cover {usd(fixed)}/month')


## 7. Sensitivity to lecture duration

The headline assumption. If real lectures are longer or shorter than 75 minutes,
this is what has to be re-decided.


In [ ]:
# Re-run the whole per-lecture calculation at several durations. If real
# lectures turn out to be 50 or 110 minutes, this is the row to read — and the
# token column shows how far the caps would have to move.
rows = []
for d in A['sensitivityDurationsMinutes']:
    rr = per_lecture(d)
    t = sum(rr['lines'].values())
    pro = A['tiers']['pro']['lecturesPerMonth'] * rr['aiTokens']  # a month of Pro use
    rows.append([f'{d} min', usd(t - rr['cloudOnly']), usd(t),
                 f'{pro/1e6:.0f}M', f"{rr['audioMb']:.0f} MB"])
table(['Duration', 'Per lecture (browser)', 'Per lecture (cloud)',
       'Pro aiTokens needed/mo', 'Audio per lecture'], rows,
      'Duration sensitivity')


## 8. Caps the assumptions imply

Derives what each cap *would* be from the usage model, so you can see where the
shipped values were rounded up for headroom — or deliberately set below full
coverage, as `sttMinutes` and `diarizationMinutes` are, because at
$0.016/min full coverage would dominate the tier.


In [ ]:
# Sanity check in the opposite direction: instead of pricing the caps we ship,
# work out what the caps SHOULD be from the usage assumptions, and compare. A
# headroom below 1.0 means the cap would bind before a normal month is over.
rows = []
for tier, cfg in A['tiers'].items():
    n = cfg['lecturesPerMonth']
    rr = per_lecture()
    derived = {
        'aiTokens': n * rr['aiTokens'],
        # A month of narration plus the re-synthesis that editing causes.
        'ttsCharacters': n * (rr['narrationChars']
                              + A['revision']['resynthesizedSlides'] * A['lecture']['narrationCharsPerSlide']),
        'audienceLocales': n * A['audience']['localesPerDeck'],
        # One image lookup per slide, plus the share of slides re-enriched.
        'imageLookups': n * rr['slides'] * (1 + A['revision']['enrichmentRedoShare']),
    }
    for metric, value in derived.items():
        shipped = PLANS[tier]['caps'].get(metric)
        rows.append([tier, metric, f'{value:,.0f}',
                     'unlimited' if shipped is None else f'{shipped:,}',
                     # A cap of 0 means the feature is off for this tier, so a
                     # ratio would be meaningless.
                     '—' if shipped in (None, 0) else f'{shipped/value:.2f}x'])
table(['Tier', 'Metric', 'Implied by assumptions', 'Shipped cap', 'Headroom'], rows,
      'Derived vs shipped')


## 9. Fiddling with this

- **Lecture length wrong?** `assumptions.lecture.durationMinutes` — everything rescales.
- **Students review more than we think?** `assumptions.audience.*`. Note playback of
  cached content is nearly free; only *new* languages spend, so `localesPerDeck` moves
  the number far more than `playbacksPerViewingStudent` does.
- **Vendor changed a price?** `pricing.json`, then re-run — the drift check in §1 will
  tell you it no longer matches what the server bills against.
- **Trying a different model?** `pricing.ai.defaultModel`. 2.5 Flash-Lite is ~2.5×
  cheaper than the current default; 3.5 Flash is ~6× dearer.
- **Considering a cap change?** Edit `../config/plans.json` and re-run §5 and §6 to see
  the effect on worst-case cost and break-even before shipping it.

When a change here should stick, update
[`docs/BILLING_COST_MODEL.md`](../docs/BILLING_COST_MODEL.md) so the prose and the
notebook agree — the doc is the explanation, this is the calculator.
